### Set up Paths

In [ ]:
! git clone https://github.com/GP2code/long-gwas-pipeline.git

In [ ]:
# Mount to working EU directory
! gcsfuse --dir-mode 777 --file-mode 777 --implicit-dirs gp2_working_eu ./working

# Mount to release bucket - read access
! gcsfuse --dir-mode 555 --file-mode 444 --implicit-dirs gp2_release9 ./rel9_info

# Mount to release bucket - read access
! gcsfuse --dir-mode 555 --file-mode 444 --implicit-dirs gp2tier2_vwb ./rel7_info

## Run Nextflow Pipeline

Run in run_pipeline/run_trial folder; may want to run in /dev/shm/ and copy over only what we want to keep -- the rest will be deleted when the VM stops

In [ ]:
anc = 'EUR'
analysis = 'SURV'
chrom_start = 5
chrom_end = 6

# ! mkdir /dev/shm/hold_data # if want to use tmp dir in VM

# for chrom in range(chrom_start,chrom_end + 1):
#     ! gsutil -m cp -r gs://gp2_working_eu/nicole/JAN/{anc.upper()}/{analysis}/chr{chrom}_hy_{analysis}_subsetted_{anc.upper()}.vcf.gz /dev/shm/hold_data/

for chrom in range(chrom_start,chrom_end + 1):
    ! gsutil -m cp -r gs://gp2_working_eu/nicole/JAN/{anc.upper()}/{analysis}/chr{chrom}_hy_{analysis}_subsetted_{anc.upper()}.vcf.gz /home/kuznetsovn2/chroms/

# ! gsutil cp gs://gp2_working_eu/nicole/JAN/{anc.upper()}/{analysis}/{anc.upper()}_{analysis}_chroms_tmp_dir.yml /home/kuznetsovn2/run_files

! gsutil cp gs://gp2_working_eu/nicole/JAN/{anc.upper()}/{analysis}/hy_{analysis}_{anc.upper()}_covariates.tsv /dev/shm/hold_data/
! gsutil cp gs://gp2_working_eu/nicole/JAN/{anc.upper()}/{analysis}/hy_{analysis}_{anc.upper()}_phenotypes.tsv /dev/shm/hold_data/

In [ ]:
anc = 'EUR'
analysis = 'SURV'
chrom_start = 4
chrom_end = 5

! chmod +x run_process.sh

# anc analysis chrom_start chrom_end tmp_dir_script trial_dir
print(f'run_process.sh {anc} {analysis} {chrom_start} {chrom_end} 1 1')

### Current Chrom 4 Fix

In [ ]:
# Run QC before split in analysis 

month = 'JAN'
label = 'EUR'
analysis = 'CS'
chrom = 4

keep_samples_path = f'working/nicole/{month}/{label}/{analysis}/{label}_keep_samples.txt'
pfile_path = f'rel9_info/imputation_out_dir/{label}/final_files/chr{chrom}_{label}_release9_vwb'
out_path = f'plink_output/chr{chrom}_hy_{analysis}_subsetted_{label}'

! plink2  --pfile {pfile_path} --keep {keep_samples_path} --make-pgen --out {out_path}_sample_subset

! plink2 --pfile {out_path}_sample_subset --make-pgen --snps-only just-acgt --mac 2 --out {out_path}_split_hg38_normalized_snps

! plink2 --pfile {out_path}_split_hg38_normalized_snps --make-pgen --geno 0.05 dosage --out {out_path}_p1out

! plink2  --pfile {out_path}_p1out --export vcf 'bgz' 'vcf-dosage=HDS' --out chroms/chr{chrom}_hy_{analysis}_subsetted_{label}

# Check missingness with MAF filter
# ! plink2  --pfile {out_path}_sample_subset --maf 0.05 --make-pgen --out {out_path}_sample_subset_maf

# ! plink2 --pfile {out_path}_sample_subset_maf --missing --out {out_path}_check_missingness_maf

In [ ]:
# Run QC before split in analysis - release 7

month = 'DEC'
label = 'EUR'
analysis = 'CS'
chrom = 4


keep_samples_path = f'working/nicole/{month}/{label}/{analysis}/{label}_keep_samples.txt'
pfile_path = f'rel7_info/release7_30042024/imputed_genotypes/{label}/chr{chrom}_{label}_release7_vwb'
out_path = f'plink_output/R7/chr{chrom}_hy_{analysis}_subsetted_{label}'

! plink2  --pfile {pfile_path} --keep {keep_samples_path} --make-pgen --out {out_path}_sample_subset

! plink2 --pfile {out_path}_sample_subset --make-pgen --snps-only just-acgt --mac 2 --out {out_path}_split_hg38_normalized_snps

! plink2 --pfile {out_path}_split_hg38_normalized_snps --make-pgen --geno 0.05 dosage --out {out_path}_p1out

# Check missingness
# ! plink2 --pfile {out_path}_sample_subset --missing --out {out_path}_check_missingness